# 🔍 RAG Pipeline — Mistral + FAISS
**Retrieval-Augmented Generation** over your PDF and Text/Markdown files.

Run the cells top-to-bottom to:
1. Install dependencies
2. Configure settings
3. Ingest your documents
4. Query with Mistral

# 1 · Install dependencies

In [1]:
pip install -q mistralai faiss-cpu sentence-transformers langchain langchain-community pypdf python-dotenv rich


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.40.0 requires opentelemetry-api==1.40.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.40.0 requires opentelemetry-semantic-conventions==0.61b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.


# 2 · Configuration


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

False

# Mistral 

In [2]:
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "G1t6ajsDVwiJxOopxJjacwbsPVBEZwUc")
MISTRAL_MODEL = "mistral-small-latest"

# Embedding

In [3]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-V2"

# FAISS Index Path

In [4]:
FAISS_INDEX_PATH = "faiss_index"

# Chunking

In [5]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64

# Retrieval

In [6]:
TOP_K = 4

# Data Directory

In [7]:
DATA_DIR = "./data"

print("Configuration Ready!")
print(f"Model: {MISTRAL_MODEL}")
print(f"Embeddings: {EMBEDDING_MODEL}")
print(f"Data Dir: {DATA_DIR}")
print(f"API Key Set: {bool(MISTRAL_API_KEY and MISTRAL_API_KEY != 'G1t6ajsDVwiJxOopxJjacwbsPVBEZwUc')}")

Configuration Ready!
Model: mistral-small-latest
Embeddings: sentence-transformers/all-MiniLM-L6-V2
Data Dir: ./data
API Key Set: False


# 3 · Ingest documents


In [8]:
from pathlib import Path
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md", ".mdx"}

data_path = Path(DATA_DIR)
os.makedirs(data_path, exist_ok = True)

files = [f for f in data_path.rglob("*") if f.suffix in SUPPORTED_EXTENSIONS]
print(f"Found {len(files)} file(s): {[f.name for f in files]}")

raw_docs = []
for file in files:
    try:
        loader = PyPDFLoader(str(file)) if file.suffix == '.pdf' else TextLoader(str(file), encoding='utf-8')
        docs = loader.load()
        for doc in docs:
            doc.metadata['source'] = file.name
        raw_docs.extend(docs)
        print(f'Loaded {file.name} ({len(docs)} pages)')
    except Exception as e:
        print(f'Failed {file.name}: {e}')

c:\Users\TUF\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\TUF\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 1 file(s): ['MachineTranslationwithAttention.pdf']
Loaded MachineTranslationwithAttention.pdf (10 pages)


# Split

In [9]:
splitter = CharacterTextSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separator="",
)

chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")
print("\n--- First chunk preview ---")
print(chunks[0].page_content[:300])

Split into 77 chunks

--- First chunk preview ---
Neural Machine Translation with Attention
Mohammad Wasil Saleem
Matrikel-Nr.: 805779
Universit¨at Potsdam
saleem1@uni-potsdam.de
Sandeep Uprety
Matrikel-Nr. 804982
Universit¨at Potsdam
uprety@uni-potsdam.de
Abstract
In recent years, the success achieved
through neural machine translation has
made it


# Embed + FAISS


In [10]:
print('Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print('Building FAISS index...')
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(FAISS_INDEX_PATH)

print(f'Index saved to {FAISS_INDEX_PATH}/')

Loading embedding model...


C:\Users\TUF\AppData\Local\Temp\ipykernel_20120\2463262877.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Building FAISS index...
Index saved to faiss_index/


# 4 · Load index

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

if not os.path.isdir(FAISS_INDEX_PATH):
    raise FileNotFoundError("Run ingest step first")

if "embeddings" not in dir():
    embeddings = HuggingFaceEmbeddings(
        model_name = EMBEDDING_MODEL,
        model_kwargs = {"device": "cpu"},
        encode_kwargs = {"normalize_embeddings": True},
    )

vectorstore = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization = True
)

print("FAISS index loaded!")

def retrieve(query: str, top_k: int = TOP_K):
    return vectorstore.similarity_search(query, k = top_k)

FAISS index loaded!


# 5 · Query

In [16]:
from mistralai.client import Mistral
from IPython.display import Markdown, display

client = Mistral(api_key=MISTRAL_API_KEY)

def ask(query, show_chunks=False):
    chunks = retrieve(query)

    if show_chunks:
        for i, doc in enumerate(chunks, 1):
            print(f"\n--- Chunk {i} ---")
            print(doc.page_content[:300])

    def build_prompt(query, chunks):
        context_blocks = [
            f"[Source {i}: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content.strip()}"
            for i, doc in enumerate(chunks, 1)
        ]
        context = '\n\n---\n\n'.join(context_blocks)

        return (
            "You are a helpful assistant. Answer using ONLY the context.\n"
            "If not found, say you don't know.\n"
            "Always cite sources.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
        )

    response = client.chat.complete(
        model=MISTRAL_MODEL,
        messages=[{"role": "user", "content": build_prompt(query, chunks)}],
    )

    answer = response.choices[0].message.content.strip()
    sources = list({doc.metadata.get('source', 'unknown') for doc in chunks})

    display(Markdown(f"### Answer\n\n{answer}"))
    print("Sources:", ", ".join(sources))

In [20]:
ask("Summarise the document")

### Answer

The document discusses **machine translation (MT)**, focusing on the use of **attention mechanisms** in neural machine translation (NMT). Key points include:

1. **Background on Machine Translation**:
   - MT translates text without human assistance while preserving meaning.
   - Early approaches (1950s) were rule-based, while later methods (Statistical Machine Translation, SMT) used statistical models trained on bilingual corpora.

2. **Attention-Based NMT**:
   - The paper builds on Bahdanau et al. (2014), which introduced the **attention mechanism** to improve translation quality by aligning source and target words dynamically.
   - The authors aim to replicate results similar to Bahdanau et al. (2014), achieving a **BLEU score of 25.37**, comparable to the original work.

3. **Evaluation Metric**:
   - BLEU (Bilingual Evaluation Understudy) is used to measure translation quality automatically.

4. **Related Work**:
   - References include Luong et al. (2015) for attention-based NMT and Papineni et al. (2002) for BLEU evaluation.

**Sources**: The document cites Bahdanau et al. (2014), Luong et al. (2015), and Papineni et al. (2002) as key references for attention mechanisms and evaluation metrics in MT.

Sources: MachineTranslationwithAttention.pdf


In [25]:
ask("The dataset used for training the model")

### Answer

The dataset used for training the model consisted of 1 million sentences, split into:
- 900K sentences for training,
- 80K for validation, and
- 20K for testing (unseen during training).

No vocabulary size limit was imposed; instead, the full vocabulary from the training data was used.
Source: [Source 1: MachineTranslationwithAttention.pdf]

Sources: MachineTranslationwithAttention.pdf


In [26]:
ask("Why are GRU units preferred over simple RNNs in this work?")

### Answer

GRU units are preferred over simple RNNs in this work because they use a gating mechanism to control memory access, which helps in capturing long-range dependencies more effectively. Unlike simple RNNs, where the entire memory state is read and updated at each step without control, GRUs employ update and reset gates to regulate the flow of information, mitigating the vanishing gradient problem and improving the model's ability to retain and utilize information from earlier parts of the sequence (Source 1, Source 4).

Sources: MachineTranslationwithAttention.pdf


# 6 · Inspect Index

In [15]:
total_vectors = vectorstore.index.ntotal
print("Index contains {total_vectors} vectors")

sample_chunks = retrieve("", top_k = 3) if total_vectors else []

for i, chunk in enumerate(sample_chunks, 1):
    print(f"\nSample {i} ({chunk.metadata.get("source")})")
    print(chunk.page_content[:200])

Index contains {total_vectors} vectors

Sample 1 (MachineTranslationwithAttention.pdf)
errienboer, C ¸ aglar
G¨ulc ¸ehre, Dzmitry Bahdanau, Fethi Bougares,

Sample 2 (MachineTranslationwithAttention.pdf)
ion , ladies and gentlemen , i would like to make three brief comments
brief comments .
Google Translate Mr. Chairman, Mr. Chairman-in-Ofﬁce of the Council, Mr. Chairman of the Com-
mittee, ladies and

Sample 3 (MachineTranslationwithAttention.pdf)
i would like once again to express my pleasure at the importance
that the community attaches to the need for innovation in europe .
Our Model that ( pl ) mr president , i would once again like to expr
